In [1]:
# !pip install mlflow

In [2]:

import sys
sys.path.append("..")

import mlflow
import mlflow.sklearn
import pandas as pd

from src.data_loader import load_and_prepare
from src.models import MODEL_REGISTRY, train_model, evaluate_model
from src.tuning import tune_model

c:\Users\Twinkle\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(mlflow.__version__)

3.14.0


In [4]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("ckd-prediction")

<Experiment: artifact_location='file:///c:/Users/Twinkle/OneDrive/Desktop/ckd-prediction-system/mlartifacts', creation_time=1784954089644, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1784954089644, lifecycle_stage='active', name='ckd-prediction', tags={}, trace_location=None, workspace='default'>

In [5]:
X_train, X_test, y_train, y_test, scaler = load_and_prepare(
    path="../data/processed/kidney_features.csv",
    strategy="smote",
)
print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (400, 26) Test: (80, 26)


In [6]:
print(mlflow.active_run())

None


In [7]:
for name in MODEL_REGISTRY:
    with mlflow.start_run(run_name=f"{name}_untuned"):
        model = train_model(name, X_train, y_train)
        scores = evaluate_model(model, X_test, y_test)

        mlflow.set_tag("stage", "untuned")
        mlflow.set_tag("model_family", name)

        mlflow.log_param("model", name)
        mlflow.log_param("resampling", "smote")

        mlflow.log_metric("accuracy", scores["accuracy"])
        mlflow.log_metric("precision", scores["precision"])
        mlflow.log_metric("recall", scores["recall"])
        mlflow.log_metric("f1", scores["f1"])
        if scores["roc_auc"] is not None:
            mlflow.log_metric("roc_auc", scores["roc_auc"])

        # serialization_format="pickle" avoids skops' UntrustedTypesFoundException
        # on XGBoost/LightGBM internals; name= replaces the deprecated artifact_path=
        mlflow.sklearn.log_model(
            model,
            name="model",
            serialization_format="pickle",
        )

print("Logged", len(MODEL_REGISTRY), "untuned runs.")

2026/07/25 10:18:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/25 10:18:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/25 10:18:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

Logged 7 untuned runs.


In [8]:
candidates = ["random_forest", "xgboost", "lightgbm"]
tuned_models = {}

for name in candidates:
    with mlflow.start_run(run_name=f"{name}_tuned"):
        search = tune_model(name, X_train, y_train, scoring="recall", n_iter=20)
        best_model = search.best_estimator_
        scores = evaluate_model(best_model, X_test, y_test)
        tuned_models[name] = best_model

        mlflow.set_tag("stage", "tuned")
        mlflow.set_tag("model_family", name)

        mlflow.log_param("model", name)
        mlflow.log_param("resampling", "smote")
        for k, v in search.best_params_.items():
            mlflow.log_param(f"best_{k}", v)

        mlflow.log_metric("accuracy", scores["accuracy"])
        mlflow.log_metric("precision", scores["precision"])
        mlflow.log_metric("recall", scores["recall"])
        mlflow.log_metric("f1", scores["f1"])
        if scores["roc_auc"] is not None:
            mlflow.log_metric("roc_auc", scores["roc_auc"])

        mlflow.sklearn.log_model(
            best_model,
            name="model",
            serialization_format="pickle",
        )

print("Logged", len(candidates), "tuned runs.")

2026/07/25 10:19:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/25 10:19:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/25 10:19:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mec

Logged 3 tuned runs.


In [21]:
import mlflow
print(mlflow.active_run())
mlflow.end_run()
print(mlflow.active_run())

None
None


In [22]:
runs_df = mlflow.search_runs(experiment_names=["ckd-prediction"])
print(len(runs_df))
print(runs_df["tags.mlflow.runName"].duplicated().any())

cols = ["tags.mlflow.runName", "tags.stage", "metrics.recall", "metrics.f1", "metrics.roc_auc"]
runs_df[cols].sort_values("metrics.recall", ascending=False)

10
False


,tags.mlflow.runName,tags.stage,metrics.recall,metrics.f1,metrics.roc_auc
1,xgboost_tuned,tuned,1.00,1.000000,1.000000
2,random_forest_tuned,tuned,1.00,0.990099,1.000000
9,logreg_untuned,untuned,1.00,1.000000,1.000000
7,random_forest_untuned,untuned,1.00,1.000000,1.000000
0,lightgbm_tuned,tuned,0.98,0.989899,1.000000
3,lightgbm_untuned,untuned,0.98,0.989899,1.000000
4,xgboost_untuned,untuned,0.98,0.989899,0.999333
8,decision_tree_untuned,untuned,0.94,0.969072,0.970000
5,svm_untuned,untuned,0.92,0.938776,0.989333
6,knn_untuned,untuned,0.86,0.924731,0.981000


In [12]:
FINAL_MODEL_NAME = "xgboost"  # must match Day 8's chosen final model

best_run = runs_df[
    (runs_df["tags.model_family"] == FINAL_MODEL_NAME) & (runs_df["tags.stage"] == "tuned")
].iloc[0]
run_id = best_run["run_id"]

model_uri = f"runs:/{run_id}/model"
registered = mlflow.register_model(model_uri, "ckd_prediction_model")
print("Registered version:", registered.version)

Successfully registered model 'ckd_prediction_model'.
2026/07/25 10:20:05 WARNING mlflow.tracking._model_registry.fluent: Run with id b226acef9f824b20943be1718ff39985 has no artifacts at artifact path 'model', registering model based on models:/m-ba3b1a8d75ff4e18b64a42e9fb921725 instead


Registered version: 1


Created version '1' of model 'ckd_prediction_model'.


In [13]:
from mlflow.tracking import MlflowClient
client = MlflowClient()

version_info = client.get_model_version("ckd_prediction_model", "1")
print("Source:", version_info.source)
print("Run ID:", version_info.run_id)

Source: models:/m-ba3b1a8d75ff4e18b64a42e9fb921725
Run ID: b226acef9f824b20943be1718ff39985


In [14]:
print("Expected run_id:", best_run["run_id"])

Expected run_id: b226acef9f824b20943be1718ff39985


In [15]:
client.set_registered_model_alias("ckd_prediction_model", "production", 1)
print("Alias set.")

Alias set.


In [16]:
production_model = mlflow.sklearn.load_model("models:/ckd_prediction_model@production")
reload_scores = evaluate_model(production_model, X_test, y_test)
print(reload_scores)

{'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'roc_auc': 1.0, 'confusion_matrix': [[30, 0], [0, 50]]}


In [23]:
import hashlib

def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

data_hash = file_hash("../data/processed/kidney_features.csv")
print("Dataset hash:", data_hash)

Dataset hash: 5ef7f533bb4400f47c84a3fc4e8d933a


In [24]:
print(best_run["run_id"])

b226acef9f824b20943be1718ff39985


In [25]:
with mlflow.start_run(run_id=best_run["run_id"]):
    mlflow.set_tag("dataset_hash", data_hash)
    mlflow.log_artifact("../data/processed/kidney_features.csv", artifact_path="dataset_snapshot")
print("Tagged dataset hash and snapshot on the final run.")

Tagged dataset hash and snapshot on the final run.


In [26]:
from src.experiment_tracking import setup_tracking
setup_tracking(tracking_uri="sqlite:///../mlflow.db")